# Perceptron Example (Wine Quality Dataset)

Here it is demonstrated how to use the `Perceptron` module from the CMOR-438 library to perform binary classification.
In this example, the Wine Quality dataset is used to train, test, and evaluate the model.

**Goal: Predict whether a wine is High Quality (score ≥ 7) based on its physicochemical properties.**

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# Add the algorithm folder and shared helpers to path
NOTEBOOK_DIR = os.path.abspath('')
sys.path.insert(0, NOTEBOOK_DIR)
sys.path.insert(0, os.path.join(NOTEBOOK_DIR, '..', '_shared'))

# Data lives two levels up from the algorithm folder
DATA_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', 'data')
# Also add the PCA module for the visualisation cell
sys.path.insert(0, os.path.join(NOTEBOOK_DIR, '..', '..', 'unsupervised_learning', 'pca'))

from perceptron import Perceptron
from pca import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv(os.path.join(DATA_DIR, 'WineQT.csv')).drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")

## 2. Preprocessing

Create a binary target, standardise features, and split 80/20.

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y_bin = (wine['quality'].values >= 7).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X, y_bin, test_size=0.2, random_state=42, stratify=y_bin)

print(f"Training samples: {X_tr.shape[0]}  |  Test samples: {X_te.shape[0]}")

## 3. Train

Train the Perceptron for up to 300 passes through the training data.

In [ ]:
p = Perceptron(learning_rate=0.01, n_iterations=300)
p.fit(X_tr, y_tr)
print(f'Accuracy: {p.accuracy(X_te, y_te):.4f}')
print(f'Epochs run: {len(p.errors_per_epoch_)}')
print(f'Final epoch errors: {p.errors_per_epoch_[-1]}')

## 4. Results and Visualisation

Two plots are produced:
- **Training errors per epoch** — misclassifications each pass through the data
- **Errors in PCA space** — test errors projected to 2D; shows where the linear boundary fails

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(p.errors_per_epoch_, color='purple', lw=1.5)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Misclassifications')
axes[0].set_title('Perceptron Training Errors per Epoch', fontweight='bold')

pca_vis = PCA(n_components=2).fit(X_tr)
X_2d_te = pca_vis.transform(X_te)
preds = p.predict(X_te)
correct = (preds == np.where(y_te==0,-1,1))
axes[1].scatter(X_2d_te[correct,0],  X_2d_te[correct,1],  c='steelblue', s=15, alpha=0.6, label='Correct')
axes[1].scatter(X_2d_te[~correct,0], X_2d_te[~correct,1], c='crimson',   s=20, alpha=0.8, marker='x', label='Wrong')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('Perceptron Errors in PCA Space', fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.show()